<a href="https://colab.research.google.com/github//2502110077-CS-1281-MLuPL/blob/main/2502110077_lab09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IT12**

In [ ]:
import os

os.makedirs("data", exist_ok=True)

records = [
    "PE-0137,25.62,DAY",
    "PE-0201,24.90,NIGHT",
    "PE-0345,25.04,DAY",
    "PE-0999,25.00,SWING",
    "PE-0120,26.50,DAY",
    "PE-0450,24.80,NIGHT",
    "PE-0670,25.30,SWING",
    "PE-0780,26.10,DAY",
    "PE-0890,24.50,NIGHT",
    "PE-0230,25.70,DAY",
    "PE-0560,25.20,SWING",
    "PE-0788,24.60,NIGHT",
    "PE-ABCD,25.50,DAY",
    "PE-0456,ABC,NIGHT",
    "PE-0789,25.00,INVALID"
]

with open("data/shift_log.txt", "w") as file:
    for record in records:
        file.write(record + "\n")

with open("data/shift_log.txt", "r") as file:
    lines = file.readlines()

print("Total records:", len(lines))
print("Log file created successfully!")

Total records: 15
Log file created successfully!


**IT10**

In [ ]:
import os
import csv
import re

os.makedirs("data", exist_ok=True)

input_file = "data/shift_log.txt"

valid_records = []
rejected_records = []

total = 0

pattern = r"^\s*(PE-\d{4})\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(DAY|NIGHT|SWING)\s*$"

# READ AND PARSE
with open(input_file, "r") as file:

    for line in file:

        line = line.strip()

        if not line:
            continue

        match = re.fullmatch(pattern, line, re.IGNORECASE)

        # VALIDATE
        if match:

            code = match.group(1).upper()
            value = float(match.group(2))
            shift = match.group(3).upper()

            valid_records.append([code, value, shift])
            total += value

        else:
            rejected_records.append([
                line,
                "Invalid format or data"
            ])

# WRITE REPORT
with open("data/report.txt", "w") as file:

    file.write("PRODUCTION REPORT\n")
    file.write("=================\n")

    file.write(
        f"Total valid records: {len(valid_records)}\n"
    )

    file.write(
        f"Total rejected records: {len(rejected_records)}\n"
    )

    file.write(f"Total value: {total:.2f}\n\n")

    file.write("VALID RECORDS\n")

    for record in valid_records:
        file.write(
            f"{record[0]} | {record[1]:.2f} | {record[2]}\n"
        )

# WRITE REJECTS CSV
with open("data/rejects.csv", "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow(["Rejected Line", "Reason"])

    writer.writerows(rejected_records)

print("Pipeline completed successfully!")
print("Valid records:", len(valid_records))
print("Rejected records:", len(rejected_records))
print("Total value:", round(total, 2))
print("Report saved in data/report.txt")
print("Rejects saved in data/rejects.csv")

Pipeline completed successfully!
Valid records: 12
Rejected records: 3
Total value: 303.26
Report saved in data/report.txt
Rejects saved in data/rejects.csv


**IT8**

In [ ]:
import re

# Regex pattern
pattern = re.compile(
    r"^\s*(PE-\d{4})\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(DAY|NIGHT|SWING)\s*$",
    re.IGNORECASE
)

def parse_record(line):

    match = pattern.fullmatch(line)

    if match:

        part_code = match.group(1).upper()
        value = float(match.group(2))
        shift = match.group(3).upper()

        return part_code, value, shift

    else:
        return None


# Test records
test_records = [
    "PE-0137,25.62,DAY",
    "PE-0201 , 24.90 , NIGHT",
    " PE-0345,25.04,DAY ",
    "PE-0999,25.00,SWING",
    "PE-ABCD,25.50,DAY",
    "PE-0456,ABC,NIGHT",
    "PE-0789,25.00,INVALID"
]

print("REGEX PARSING RESULTS")
print("---------------------")

for line in test_records:

    result = parse_record(line)

    if result:
        print("VALID:", result)
    else:
        print("REJECTED:", line)

REGEX PARSING RESULTS
---------------------
VALID: ('PE-0137', 25.62, 'DAY')
VALID: ('PE-0201', 24.9, 'NIGHT')
VALID: ('PE-0345', 25.04, 'DAY')
VALID: ('PE-0999', 25.0, 'SWING')
REJECTED: PE-ABCD,25.50,DAY
REJECTED: PE-0456,ABC,NIGHT
REJECTED: PE-0789,25.00,INVALID


**IT6**

In [ ]:
import os
import csv
import re
from datetime import datetime

# Data folder
os.makedirs("data", exist_ok=True)

# Regex pattern
pattern = re.compile(
    r"^\s*(PE-\d{4})\s*,\s*(\d+(?:\.\d+)?)\s*,\s*(DAY|NIGHT|SWING)\s*$",
    re.IGNORECASE
)


def process_file(filepath):

    valid_records = []
    rejected_records = []
    total = 0

    with open(filepath, "r") as file:

        for line in file:

            line = line.strip()

            if not line:
                continue

            match = pattern.fullmatch(line)

            if match:

                code = match.group(1).upper()
                value = float(match.group(2))
                shift = match.group(3).upper()

                valid_records.append([code, value, shift])
                total += value

            else:

                rejected_records.append([
                    line,
                    "Invalid format or data"
                ])

    # Generate report for each file
    report_name = os.path.splitext(filepath)[0] + "_report.txt"

    with open(report_name, "w") as report:

        report.write("PRODUCTION REPORT\n")
        report.write("=================\n")

        report.write(
            f"Valid records: {len(valid_records)}\n"
        )

        report.write(
            f"Rejected records: {len(rejected_records)}\n"
        )

        report.write(f"Total value: {total:.2f}\n")

    return len(valid_records), len(rejected_records), total


# BATCH PROCESSING

files = [
    f for f in os.listdir("data")
    if f.endswith(".txt")
    and not f.endswith("_report.txt")
]

history = []
combined_total = 0
combined_rejected = 0
combined_valid = 0

for filename in files:

    filepath = os.path.join("data", filename)

    valid, rejected, total = process_file(filepath)

    combined_valid += valid
    combined_rejected += rejected
    combined_total += total

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    history.append([
        timestamp,
        filename,
        valid,
        rejected,
        total
    ])

    print("Processed:", filename)


# RUN HISTORY

with open("data/run_history.csv", "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "Timestamp",
        "Filename",
        "Valid Records",
        "Rejected Records",
        "Total Value"
    ])

    writer.writerows(history)


# COMBINED PLANT SUMMARY

with open("data/plant_summary.txt", "w") as file:

    file.write("COMBINED PLANT SUMMARY\n")
    file.write("======================\n")

    file.write(f"Files processed: {len(files)}\n")
    file.write(f"Total valid records: {combined_valid}\n")
    file.write(f"Total rejected records: {combined_rejected}\n")
    file.write(f"Combined total: {combined_total:.2f}\n")


print("\nBATCH JOB COMPLETED!")
print("Files processed:", len(files))
print("Valid records:", combined_valid)
print("Rejected records:", combined_rejected)
print("Combined total:", round(combined_total, 2))
print("History saved: data/run_history.csv")
print("Plant summary saved: data/plant_summary.txt")

Processed: shift_log.txt
Processed: report.txt

BATCH JOB COMPLETED!
Files processed: 2
Valid records: 12
Rejected records: 21
Combined total: 303.26
History saved: data/run_history.csv
Plant summary saved: data/plant_summary.txt
